# Inspect UMM Tokenizer Codebook
2JUL2024 @hanoihantrakul
This notebook helps you plot the codes in the codebook of a tokenizer. You can plot and visualize these directly because all continuous embeddings will be quantized to one of these codebook entries. The trained codebook are the like the vector centroids of a K-means clustering algorithm, which you can investigate separately without passing any data into the model.

I was using this notebook to mainly investigate the appearance of zero-magnitude codes. These are entries in the codebook where all dimensions are 0 and thus the overall vector magnitude or vector norm is 0. In some cases like the legacy_ConformerUMM used in V3.5 L2S Product Model, I found ~27% of the codebook was actually zero-magnitude. 

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch
import torchaudio
import IPython.display as ipd
assert torch.cuda.is_available()
import sys
import os
import numpy as np

%matplotlib inline
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
os.chdir('/opt/tiger/samantha')

In [ ]:
from recipes.umm.models.dualumm_vector_quantizers import get_vq_codebook_distances
def get_codebook_magnitudes(codebook):
    return codebook.norm(p=2,dim=1)

# Define Models

In [ ]:
from recipes.umm.requires.model_initializer import (init_stage3_conv1d, init_stage3, init_stage2, init_convumm_gan)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def load_model_convumm_gan(ckpt_path, cache_dir):
    print(f"Downloading {ckpt_path}")
    DUMMY_RANK = 0
    token_model = init_convumm_gan(ckpt_path, DUMMY_RANK, cache_dir)[
        "Stage3"
    ].eval()
    return token_model

def load_model_convumm_conv1D(ckpt_path, cache_dir):
    print(f"Downloading {ckpt_path}")
    DUMMY_RANK = 0
    token_model = init_stage3_conv1d(ckpt_path, DUMMY_RANK, cache_dir)[
        "Stage3Conv1D"
    ].eval()
    return token_model

def load_model_conformer(ckpt_path, cache_dir):
    print(f"Downloading {ckpt_path}")
    DUMMY_RANK = 0
    token_model = init_stage3(ckpt_path, DUMMY_RANK, cache_dir)[
        "Stage3"
    ].eval()
    return token_model

def load_model_conformer_stage2(ckpt_path, cache_dir):
    print(f"Downloading {ckpt_path}")
    DUMMY_RANK = 0
    token_model = init_stage2(ckpt_path, DUMMY_RANK, cache_dir)[
        "Stage2"
    ].eval()
    return token_model

def load_model_convumm_pitch(ckpt_path, cache_dir):
    print(f"Downloading {ckpt_path}")
    DUMMY_RANK = 0
    token_model = init_stage3_conv1d(ckpt_path, DUMMY_RANK, cache_dir)[
        "Stage3Conv1D"
    ].eval()
    return token_model 

MODELS_DICT = {
    # GROUP A
    # @hanoihantrakul: 2JUN2024 This is the latest ConvUMM-GAN trained on 2255 2375. These ones incorrectly used LAS loss and are depracted in master branch.
    "convumm_gan_2255_2375": "hdfs://haruna/home/byte_data_seed/lf_lq/speech/user/hanoi.hantrakul/logs/convumm_gan_master/convumm_gan_719M_2255_2375_vocals_bert-base-multilingual-uncased_EMAEntropy32768x32/checkpoints/step=0590000.ckpt",
    "convumm_gan_2255_2375_step100k": "hdfs://haruna/home/byte_data_seed/lf_lq/speech/user/hanoi.hantrakul/logs/convumm_gan_master/convumm_gan_719M_2255_2375_vocals_bert-base-multilingual-uncased_EMAEntropy32768x32/checkpoints/step=0100000.ckpt",
    "convumm_gan_2255_2375_step10k": "hdfs://haruna/home/byte_data_seed/lf_lq/speech/user/hanoi.hantrakul/logs/convumm_gan_master/convumm_gan_719M_2255_2375_vocals_bert-base-multilingual-uncased_EMAEntropy32768x32/checkpoints/step=0010000.ckpt",
    
    # GROUP B
    # @hanoihantrakul: 22APR2024 This is the latest Direct Stage3 ConvUMM trained on 2255 2375
    "convumm_direct_stage3_2255_2375": "hdfs://haruna/home/byte_data_seed/lf_lq/speech/user/hanoi.hantrakul/logs/umm_conv_mixedZHEN/umm_stage3_conv1D_v2_2250mixedZHEN_2375Speech_direct_stage3_optim_mem_EMAVQ32768x32/checkpoints/step=0310000.ckpt",
    "convumm_direct_stage3_2255_2375_step100k": "hdfs://haruna/home/byte_data_seed/lf_lq/speech/user/hanoi.hantrakul/logs/umm_conv_mixedZHEN/umm_stage3_conv1D_v2_2250mixedZHEN_2375Speech_direct_stage3_optim_mem_EMAVQ32768x32/checkpoints/step=0100000.ckpt",
    "convumm_direct_stage3_2255_2375_step10k": "hdfs://haruna/home/byte_data_seed/lf_lq/speech/user/hanoi.hantrakul/logs/umm_conv_mixedZHEN/umm_stage3_conv1D_v2_2250mixedZHEN_2375Speech_direct_stage3_optim_mem_EMAVQ32768x32/checkpoints/step=0010000.ckpt",
    # @hanoihantrakul: 26APR2024 This is the latest ConformerUMM trained on 2255 2375
    "conformerumm_2255_2375": "hdfs://haruna/home/byte_data_seed/lf_lq/speech/user/hanoi.hantrakul/logs/umm_conformer_2255mixedZHEN_2375Speech/umm_stage3_2255mixedZHEN_2375Speech_optim_mem_bert-base-multilingual-uncased_None32768x32/checkpoints/step=0180000.ckpt",
    "conformerumm_2255_2375_step10k": "hdfs://haruna/home/byte_data_seed/lf_lq/speech/user/hanoi.hantrakul/logs/umm_conformer_2255mixedZHEN_2375Speech/umm_stage3_2255mixedZHEN_2375Speech_optim_mem_bert-base-multilingual-uncased_None32768x32/checkpoints/step=0010000.ckpt",
    
    # Group C
    # @hanoihantrakul: OCT2023 Legacy ConformerUMM that was reference baseline for QQ L2S ZH. Remember though that his was trained with x4 instances.
    "legacy_conformerumm_l2s_zh": "hdfs://haruna/home/byte_speech_sv/zongyu.yin/logs/umm/umm_stage3_zh_dw1-1-0_wordpiece_vq32768x16-layer12/checkpoints/step=070000.ckpt",
    "legacy_conformerumm_l2s_zh_step10k": "hdfs://haruna/home/byte_speech_sv/zongyu.yin/logs/umm/umm_stage3_zh_dw1-1-0_wordpiece_vq32768x16-layer12/checkpoints/step=010000.ckpt",
    # @hanoihantrakul: 3JUL2024 I never trained this model, so I can only guess this was the Stage2 model used to train the Stage3 model.
    "legacy_conformerumm_l2s_zh_stage2": "hdfs://haruna/home/byte_speech_sv/zongyu.yin/logs/umm/umm_stage2_zh_dw1-1-0_wordpiece/checkpoints/step=040000.ckpt",
    # @hanoihantrakul: OCT2023 Legacy ConformerUMM that was reference baseline for QQ L2S ZH
    "legacy_conformerumm_svs": "hdfs:///home/byte_speech_sv/zongyu.yin/logs/umm_mix/umm_stage3_preclipped_bert-base-multilingual-uncased_None32768x32/checkpoints/step=0330000.ckpt",
    "legacy_conformerumm_svs_step100k": "hdfs:///home/byte_speech_sv/zongyu.yin/logs/umm_mix/umm_stage3_preclipped_bert-base-multilingual-uncased_None32768x32/checkpoints/step=0100000.ckpt",
    "legacy_conformerumm_svs_step10k": "hdfs:///home/byte_speech_sv/zongyu.yin/logs/umm_mix/umm_stage3_preclipped_bert-base-multilingual-uncased_None32768x32/checkpoints/step=0010000.ckpt",
    
    # Group D
    # @hanoihantrakul: 2JUNE2024 Supervised Pitch Loss (SPL)
    "convumm_spl": "hdfs://haruna/home/byte_data_seed/lf_lq/speech/user/hanoi.hantrakul/logs/umm_conv_pitch_losses/umm_direct_stage3_conv1d_2255_2375_supervised_pitch_loss_EMAVQ32768x32/checkpoints/step=0500000.ckpt",
    # @hanoihantrakul: 2JUNE2024 Perceptual Pitch Loss (PPL)
    "convumm_ppl": "hdfs://haruna/home/byte_data_seed/lf_lq/speech/user/hanoi.hantrakul/logs/umm_conv_pitch_losses/umm_direct_stage3_conv1d_2255_2375_perceptual_pitch_loss_EMAVQ32768x32/checkpoints/step=0450000.ckpt",
    # @hanoihantrakul: 2JUNE2024 Supervised + Perceptual Pitch Loss (SPL+PPL)
    "convumm_spl_ppl": "hdfs://haruna/home/byte_data_seed/lf_lq/speech/user/hanoi.hantrakul/logs/umm_conv_pitch_losses/umm_direct_stage3_conv1d_2255_2375_supervised_and_perceptual_pitch_loss_EMAVQ32768x32/checkpoints/step=0470000.ckpt",
    "convumm_spl_ppl_step100k": "hdfs://haruna/home/byte_data_seed/lf_lq/speech/user/hanoi.hantrakul/logs/umm_conv_pitch_losses/umm_direct_stage3_conv1d_2255_2375_supervised_and_perceptual_pitch_loss_EMAVQ32768x32/checkpoints/step=0100000.ckpt",
    "convumm_spl_ppl_step10k": "hdfs://haruna/home/byte_data_seed/lf_lq/speech/user/hanoi.hantrakul/logs/umm_conv_pitch_losses/umm_direct_stage3_conv1d_2255_2375_supervised_and_perceptual_pitch_loss_EMAVQ32768x32/checkpoints/step=0010000.ckpt",
    
    # Group E
    # @hanoihantrakul: 1JUL2024 ConvUMM-GAN CTC 
    "convumm_gan_ctc1": "hdfs://haruna/home/byte_data_seed/lf_lq/speech/user/hanoi.hantrakul/logs/convumm_gan_master/convumm_gan_766M_2255_2375_vocals_w_loss_ctc_1/checkpoints/step=0440000.ckpt",
    "convumm_gan_ctc1_step100k": "hdfs://haruna/home/byte_data_seed/lf_lq/speech/user/hanoi.hantrakul/logs/convumm_gan_master/convumm_gan_766M_2255_2375_vocals_w_loss_ctc_1/checkpoints/step=0100000.ckpt",
    "convumm_gan_ctc1_step10k": "hdfs://haruna/home/byte_data_seed/lf_lq/speech/user/hanoi.hantrakul/logs/convumm_gan_master/convumm_gan_766M_2255_2375_vocals_w_loss_ctc_1/checkpoints/step=0010000.ckpt",
    
    
    # Group 1
    # @hanoihantrakul: 20MAY2024 This is the latest ConvUMM-GAN trained on SSTK instrumentals with better E2E performance compared to ConvUMM and ConformerUMM for instrumental pipeline.
    "convumm_gan_sstk_v9_product": "hdfs://haruna/home/byte_data_seed/lf_lq/speech/user/hanoi.hantrakul/logs/convumm_gan_master/convumm_gan_719M_sstk_None_EMAEntropy32768x32/checkpoints/step=0700000.ckpt",
    "convumm_gan_sstk_v9_product_step10k": "hdfs://haruna/home/byte_data_seed/lf_lq/speech/user/hanoi.hantrakul/logs/convumm_gan_master/convumm_gan_719M_sstk_None_EMAEntropy32768x32/checkpoints/step=0010000.ckpt",
    "convumm_gan_sstk_v9_product_step100k": "hdfs://haruna/home/byte_data_seed/lf_lq/speech/user/hanoi.hantrakul/logs/convumm_gan_master/convumm_gan_719M_sstk_None_EMAEntropy32768x32/checkpoints/step=0100000.ckpt",
    "convumm_gan_sstk_v9_product_step400k": "hdfs://haruna/home/byte_data_seed/lf_lq/speech/user/hanoi.hantrakul/logs/convumm_gan_master/convumm_gan_719M_sstk_None_EMAEntropy32768x32/checkpoints/step=0400000.ckpt",
    "conformer_sstk_locality_issue": "hdfs://haruna/home/byte_speech_sv/zongyu.yin/logs/umm_sstk/umm_stage3_None_EMAVQ32768x32_only_sstk_wo_ctc/checkpoints/step=0600000.ckpt",
    "conformer_sstk_locality_issue_step10k": "hdfs://haruna/home/byte_speech_sv/zongyu.yin/logs/umm_sstk/umm_stage3_None_EMAVQ32768x32_only_sstk_wo_ctc/checkpoints/step=0010000.ckpt",
    "conformer_sstk_locality_issue_step100k": "hdfs://haruna/home/byte_speech_sv/zongyu.yin/logs/umm_sstk/umm_stage3_None_EMAVQ32768x32_only_sstk_wo_ctc/checkpoints/step=0100000.ckpt",
    "conformer_sstk_locality_issue_step400k": "hdfs://haruna/home/byte_speech_sv/zongyu.yin/logs/umm_sstk/umm_stage3_None_EMAVQ32768x32_only_sstk_wo_ctc/checkpoints/step=0400000.ckpt",
    # @hanoihantrakul: these are the direct stage3 models
    "convumm_sstk_direct_stage3": "hdfs://haruna/home/byte_speech_sv/hanoi.hantrakul/logs/umm_conv_sstk/umm_stage3_conv1D_v2_733Mparams_sstk_no_ctc_794_direct_stage3_EMAVQ32768x32/checkpoints/step=0110000.ckpt",
}

# Inspect UMM Tokenizer

In [ ]:
# Select the model key based on the dictionary above

MODEL_KEY = "conformer_sstk_locality_issue"
ckpt_path = MODELS_DICT[MODEL_KEY]
unique_cache_dir = "./." + MODEL_KEY

In [ ]:
# Load model. Uncomment the one you want

load_func = load_model_convumm_gan
#load_func = load_model_conformer
#load_func = load_model_convumm_conv1D
#load_func = load_model_conformer_stage2

token_model = load_func(ckpt_path, unique_cache_dir)

In [ ]:
#codebook = token_model.vq.embedding.weight # for convumm-gan variants
codebook = token_model.model.vq.embedding.weight # for conformerumm, convumm, convumm-pitch variants
print(codebook.shape)

In [ ]:
# Probe the codebook statistics

codebook_stats = get_vq_codebook_distances(codebook)
for k, v in codebook_stats.items():
    print(f"{k}: {v:.3f}")
    
codebook_mags = get_codebook_magnitudes(codebook)
codebook_mags = codebook_mags.cpu().detach().numpy()
print(codebook_mags.shape)

In [ ]:
# Plot the distribution

NUM_BINS=500
plt.hist(codebook_mags, bins=NUM_BINS, color='skyblue', edgecolor='black')
plt.xlabel('Codebook Magnitude')
plt.ylabel('Counts')
plt.title(f"{MODEL_KEY} \n Codebook Magnitude Distribution. Num bins: {NUM_BINS}")
plt.show()

In [ ]:
# Get the unique values and their counts

unique_values, counts = np.unique(codebook_mags, return_counts=True)

# Print the results
cnt = 0
for value, count in zip(unique_values, counts):
    print(f"Vector magnitude value: {value:.6f} occurs {count} times")
    cnt+=1
    if cnt > 10:
        break

In [ ]:
# Find where the zero-tokens exist and get their specific ID's

zero_mag_idxs = np.where(codebook_mags == 0)[0]
print(zero_mag_idxs.shape)
print(zero_mag_idxs)

# Save these zero-mag indexes
# np.save('./legacy_conformerumm_l2s_zh_zero_mag_idxs', zero_mag_idxs)

In [ ]:
# Print out the first 10 tokens and their counts

# for i in range(10):
#     print(f"\n Token ID: {zero_mag_idxs[i]}")
#     print(codebook[zero_mag_idxs[i]])

# Load existing statisticsfrom numpy import linalg as LA

In [ ]:
from numpy import linalg as LA

In [ ]:
BASE_DIR = "hdfs://haruna/home/byte_speech_sv/hanoi.hantrakul/logs/umm_conv_sstk/umm_stage3_conv1D_v2_733Mparams_sstk_no_ctc_794_direct_stage3_EMAVQ32768x32/checkpoints"
STEP_CKPTS = ["step=0110000", "step=0090000", "step=0070000", "step=0050000", "step=0030000", "step=0010000"]
SAVE_DIR = "umm_stage3_conv1D_v2_733Mparams_sstk_no_ctc_794_direct_stage3_EMAVQ32768x32"

os.makedirs(SAVE_DIR, exist_ok=True)

for ckpt in STEP_CKPTS:
    ckpt_path = f"{BASE_DIR}/{ckpt}.ckpt"
    print(ckpt_path)
    load_func = load_model_conformer
    #load_func = load_model_convumm_conv1D
    #load_func = load_model_convumm_gan
    
    token_model = load_func(ckpt_path, unique_cache_dir)
    codebook = token_model.model.vq.embedding.weight # weirdly needed this for "conformer_sstk_locality_issue"
    #codebook = token_model.vq.embedding.weight
    print(codebook.shape)

    codebook = codebook.cpu().detach().numpy()
    np.save(os.path.join(SAVE_DIR, ckpt), codebook)

In [ ]:
# This is a self-enclosed cell for analyzing the npy file saved from the cells above

# BASE_DIR = "umm_stage3_zh_dw1-1-0_wordpiece_vq32768x16-layer12_codebook"
# step10k = np.load(os.path.join(BASE_DIR,"step=010000.npy"))
# step20k = np.load(os.path.join(BASE_DIR,"step=020000.npy"))
# step30k = np.load(os.path.join(BASE_DIR,"step=030000.npy"))
# step40k = np.load(os.path.join(BASE_DIR,"step=040000.npy"))
# step50k = np.load(os.path.join(BASE_DIR,"step=050000.npy"))
# step60k = np.load(os.path.join(BASE_DIR,"step=060000.npy"))
# step70k = np.load(os.path.join(BASE_DIR,"step=070000.npy"))

# step10k_mag = LA.norm(step10k, axis=1)
# step20k_mag = LA.norm(step20k, axis=1)
# step30k_mag = LA.norm(step30k, axis=1)
# step40k_mag = LA.norm(step40k, axis=1)
# step50k_mag = LA.norm(step50k, axis=1)
# step60k_mag = LA.norm(step60k, axis=1)
# step70k_mag = LA.norm(step70k, axis=1)
# all_mags = [step10k_mag, step20k_mag, step30k_mag, step40k_mag, step50k_mag, step60k_mag, step70k_mag]
# mag_max = np.max(np.concatenate(all_mags, axis=0))
# mag_min = np.min(np.concatenate(all_mags, axis=0))

# NUM_BINS=100
# MODEL_KEY="legacy_conformerumm_l2s_zh"
# STEPS = ["10k","20k","30k","40k","50k","60k","70k"]
# for idx, step_mag in enumerate(all_mags):
#     plt.figure()
#     plt.hist(step_mag, bins=NUM_BINS, color='skyblue', edgecolor='black')
#     plt.xlabel('Codebook Magnitude')
#     plt.ylabel('Counts')
#     plt.xlim(0, mag_max)
#     plt.title(f"Codebook Magnitude Distribution.\n {MODEL_KEY}\n Step:{STEPS[idx]} \nNum bins: {NUM_BINS}")
#     plt.show()

In [ ]:
# This is a self-enclosed cell for analyzing the npy file saved from the cells above

BASE_DIR = "umm_stage3_conv1D_v2_733Mparams_sstk_no_ctc_794_direct_stage3_EMAVQ32768x32"
step_codebooks = []
step_codebooks_mags = []
STEP_CKPTS = ["step=0110000", "step=0090000", "step=0070000", "step=0050000", "step=0030000", "step=0010000"]

for step_ckpt in STEP_CKPTS:
    step_codebook = np.load(os.path.join(BASE_DIR,f"{step_ckpt}.npy"))
    step_codebooks.append(step_codebook)
    step_codebooks_mags.append(LA.norm(step_codebook, axis=1))

mag_max = np.max(np.concatenate(step_codebooks_mags, axis=0))
mag_min = np.min(np.concatenate(step_codebooks_mags, axis=0))

NUM_BINS=100
MODEL_KEY="umm_stage3_conv1D_v2_733Mparams_sstk_no_ctc_794_direct_stage3_EMAVQ32768x32"
for idx, step_mag in enumerate(step_codebooks_mags):
    plt.figure()
    plt.hist(step_mag, bins=NUM_BINS, color='skyblue', edgecolor='black')
    plt.xlabel('Codebook Magnitude')
    plt.ylabel('Counts')
    plt.xlim(0, mag_max)
    plt.ylim(0, 2000)
    plt.title(f"Codebook Magnitude Distribution.\n {MODEL_KEY}\n Step:{STEP_CKPTS[idx]} \nNum bins: {NUM_BINS}")
    plt.show()

In [ ]:
# This is a self-enclosed cell for analyzing the npy file saved from the cells above

# BASE_DIR = "umm_stage3_2255mixedZHEN_2375Speech_optim_mem_bert-base-multilingual-uncased_None32768x32"
# step_codebooks = []
# step_codebooks_mags = []
# STEP_CKPTS = ["step=0180000", "step=0150000", "step=0120000", "step=0090000", "step=0060000", "step=0030000"]

# for step_ckpt in STEP_CKPTS:
#     step_codebook = np.load(os.path.join(BASE_DIR,f"{step_ckpt}.npy"))
#     step_codebooks.append(step_codebook)
#     step_codebooks_mags.append(LA.norm(step_codebook, axis=1))

# mag_max = np.max(np.concatenate(step_codebooks_mags, axis=0))
# mag_min = np.min(np.concatenate(step_codebooks_mags, axis=0))

# NUM_BINS=100
# MODEL_KEY="umm_stage3_2255mixedZHEN_2375Speech_optim_mem_bert-base-multilingual-uncased_None32768x32"
# for idx, step_mag in enumerate(step_codebooks_mags):
#     plt.figure()
#     plt.hist(step_mag, bins=NUM_BINS, color='skyblue', edgecolor='black')
#     plt.xlabel('Codebook Magnitude')
#     plt.ylabel('Counts')
#     plt.xlim(0, mag_max)
#     plt.ylim(0, 2000)
#     plt.title(f"Codebook Magnitude Distribution.\n {MODEL_KEY}\n Step:{STEP_CKPTS[idx]} \nNum bins: {NUM_BINS}")
#     plt.show()

In [ ]:
# This is a self-enclosed cell for analyzing the npy file saved from the cells above

# BASE_DIR = "convumm_gan_719M_sstk_None_EMAEntropy32768x32"
# step_codebooks = []
# step_codebooks_mags = []
# STEP_CKPTS = ["step=0700000", "step=0650000", "step=0600000", "step=0550000", "step=0500000", "step=0450000", "step=0400000", "step=0350000", "step=0300000", "step=0250000", "step=0200000", "step=0150000", "step=0100000", "step=0050000"]

# for step_ckpt in STEP_CKPTS:
#     step_codebook = np.load(os.path.join(BASE_DIR,f"{step_ckpt}.npy"))
#     step_codebooks.append(step_codebook)
#     step_codebooks_mags.append(LA.norm(step_codebook, axis=1))

# mag_max = np.max(np.concatenate(step_codebooks_mags, axis=0))
# mag_min = np.min(np.concatenate(step_codebooks_mags, axis=0))

# NUM_BINS=100
# MODEL_KEY="convumm_gan_719M_sstk_None_EMAEntropy32768x32"
# for idx, step_mag in enumerate(step_codebooks_mags):
#     plt.figure()
#     plt.hist(step_mag, bins=NUM_BINS, color='skyblue', edgecolor='black')
#     plt.xlabel('Codebook Magnitude')
#     plt.ylabel('Counts')
#     plt.xlim(0, mag_max-30)
#     plt.ylim(0, 7000)
#     plt.title(f"Codebook Magnitude Distribution.\n {MODEL_KEY}\n Step:{STEP_CKPTS[idx]} \nNum bins: {NUM_BINS}")
#     plt.show()